# 교시별 실습 Notebook

d1-p04-05

수업 페이지의 순서대로 실행합니다. 정답 블록과 선택 웹 자동화는 설명을 읽고 별도로 실행하세요.

In [ ]:
from pathlib import Path
import os

candidates = [Path.cwd(), *Path.cwd().parents]
course_root = next((p for p in candidates if (p / "raw/기존강사 수업자료").is_dir()), None)
if course_root is None:
    raise FileNotFoundError("수업 자료를 풀고 raw 폴더가 있는 위치에서 Notebook을 여세요.")
os.chdir(course_root)
print("수업 자료 폴더를 확인했습니다.")

# 요약

CSV·Excel을 읽고 필요한 행과 열을 고른 뒤 결측치를 확인합니다. 4~5교시는 같은 거래 표를 연속해서 다루므로 한 페이지로 묶었습니다.

## 수업 위치와 목표

**2026년 10월 6일 · 1일차 4~5교시 · 1차 초안.** 공식 「데이터 수집 기초 및 pandas 활용」 3시간 중 파일·표 처리에 2교시, 웹 수집에 1교시를 쓰는 제안입니다. 공식 교시별 배분은 없습니다.

파일 읽기 결과의 행·열·자료형을 확인하고, `loc`·`iloc`·조건식을 사용하며, 결측치 처리에 따라 합계가 바뀌는 이유를 설명하는 것이 목표입니다.

## 교시별 진행

| 교시 | 진행 순서 | 확인할 결과 |
| --- | --- | --- |
| 4교시 | 작업 폴더 → CSV·Excel 저장·읽기 → head·shape·dtypes → 열과 행 선택 | 두 파일에서 같은 4행·3열 표 확인 |
| 5교시 | 복합 조건 → 결측치 개수 → 중앙값 대체·행 제외 비교 → 결과 저장 | 130000과 118000의 차이를 설명 |

새 Notebook `02-pandas-collection.ipynb`를 수업 폴더에 저장합니다. 아래 코드는 연습용 `practice` 폴더를 만들며 원자료를 수정하지 않습니다. Excel 읽기·쓰기에 `openpyxl`이 필요합니다. 없다면 Anaconda Prompt에서 수업 환경을 활성화한 후 `conda install openpyxl`을 실행합니다.

## 1. CSV, Excel 파일 읽기 및 pandas 기본 사용법

CSV는 구분자로 열을 나눈 텍스트이고, Excel 파일은 시트가 있는 통합 문서입니다. 파일을 읽으면 두 형식 모두 DataFrame으로 다룰 수 있습니다. `index=False`는 저장할 때 행 번호를 데이터 열로 넣지 않는 옵션입니다.

In [ ]:
from pathlib import Path
import pandas as pd

work = Path("practice")
work.mkdir(exist_ok=True)
orders = pd.DataFrame({
    "product": ["A", "B", "A", "C"],
    "price": [10000, 20000, 12000, None],
    "quantity": [3, 2, 4, 1],
})
orders.to_csv(work / "orders.csv", index=False, encoding="utf-8-sig")
orders.to_excel(work / "orders.xlsx", index=False, sheet_name="orders")
csv_df = pd.read_csv(work / "orders.csv")
excel_df = pd.read_excel(work / "orders.xlsx", sheet_name="orders")
print("CSV:", csv_df.shape, "Excel:", excel_df.shape)
print(csv_df.head().to_string(index=False))
print(csv_df.dtypes)

두 표 모두 `(4, 3)`입니다. 마지막 가격은 비어 있고 `NaN`으로 읽힙니다. `head()`는 앞부분, `shape`는 행·열 개수, `dtypes`는 각 열의 자료형을 보여줍니다. 한글 CSV를 읽을 때 문자가 깨지면 제공 파일의 실제 인코딩을 확인합니다. 파일을 찾지 못하면 `Path.cwd()`로 Notebook의 현재 폴더를 확인합니다.

**직접 실습:** 가상 거래 한 행을 추가해 파일을 다시 저장하고, 행 수가 5로 바뀌는지 확인합니다. 다음 예제와 결과를 비교할 때는 원래 4행으로 되돌립니다.

## 2. 행/열 선택 및 조건 필터링

`df["price"]`는 한 열, `df[["product", "price"]]`는 두 열의 표입니다. `loc`는 행의 조건과 열 이름을 함께 지정하고, `iloc`는 행·열의 위치를 지정합니다. 조건 여러 개를 결합할 때는 각각 괄호로 감싸고 `&` 또는 `|`를 사용합니다.

In [ ]:
print(csv_df[["product", "price"]])
print(csv_df.iloc[:2])
selected = csv_df.loc[
    (csv_df["price"] >= 12000) & (csv_df["quantity"] >= 2),
    ["product", "price", "quantity"],
]
print(selected.to_string(index=False))

선택 결과는 B(20,000원·2개), A(12,000원·4개)의 두 행입니다. 원래 `csv_df`의 행 수는 그대로입니다. `loc`로 고른 부분을 별도로 수정하려면 `.copy()`를 붙입니다.

**직접 실습:** 상품 A이거나 수량이 1인 행을 고릅니다. 조건을 만족하는 행은 3개입니다.

## 3. 결측치 확인 및 기본 데이터 정리

값이 비어 있다는 사실과 값이 0이라는 사실은 다릅니다. 결측치를 채우는 기준은 분석 목적에 따라 선택합니다. 여기서는 연산을 배우기 위해 가격 중앙값으로 채우며, 실제 가격을 알아냈다고 해석하지 않습니다. 모델 학습용 결측치 처리는 5단원에서 훈련 데이터만 사용합니다.

In [ ]:
print(csv_df.isna().sum())
clean = csv_df.copy()
median_price = clean["price"].median()
clean["price"] = clean["price"].fillna(median_price)
clean["amount"] = clean["price"] * clean["quantity"]
print("채운 가격:", median_price)
print("총 금액:", clean["amount"].sum())
print(clean.groupby("product")["amount"].sum())

가격의 결측치는 1개, 중앙값은 12,000, 합계는 130,000입니다. 상품별 금액은 A 78,000, B 40,000, C 12,000입니다. `dropna(subset=["price"])`로 결측 행을 제외하면 합계는 118,000이 됩니다. 두 합계가 달라지는 이유를 설명합니다.

**직접 실습:** 가격 결측 행을 제외한 결과를 `without_missing`에 저장합니다. `.copy()`를 사용하고 `amount`를 계산해 합계를 비교합니다.

## 선택 결과의 형태 읽기

In [ ]:
print(type(csv_df["price"]).__name__)
print(type(csv_df[["price"]]).__name__)
print(csv_df["price"].shape)
print(csv_df[["price"]].shape)

출력은 `Series`, `DataFrame`, `(4,)`, `(4, 1)`입니다. 대괄호 하나로 열 이름 하나를 전달하면 1차원 Series가 나오고, 열 이름 리스트를 전달하면 2차원 표가 유지됩니다. 화면이 비슷하게 보여도 구조는 다릅니다.

`iloc[:2]`는 처음 두 행입니다. 슬라이스의 끝 위치 2는 포함하지 않습니다. `loc[0:2]`는 현재 인덱스가 0·1·2로 붙어 있을 때 라벨 2까지 포함합니다. 필터링한 뒤에도 인덱스가 자동으로 0부터 다시 붙지 않을 수 있으므로 인덱스와 위치를 구분합니다.

In [ ]:
filtered = csv_df.loc[csv_df["price"] >= 12000].copy()
print("필터 후 인덱스:", filtered.index.tolist())
print(filtered.iloc[0].to_dict())

인덱스는 `[1, 2]`입니다. `filtered.iloc[0]`은 그 표의 첫 번째 행이며 상품 B입니다. `filtered.loc[0]`을 쓰면 라벨 0이 없어 오류가 납니다.

## 직접 실습 — 조건 선택과 결측 처리 비교

예제를 초기 4행 상태로 실행한 뒤 다음 문제를 풉니다.

1. 상품이 A이거나 수량이 1인 행을 선택하세요.
2. 가격이 없는 행을 제외한 복사본 `without_missing`을 만드세요.
3. 복사본에 `amount`를 계산하고 합계를 출력하세요.
4. 중앙값으로 채운 `clean`의 합계와 비교하여 차이를 설명하세요.
5. `clean`을 `practice/orders_clean.csv`에 인덱스 없이 저장하세요.

<details>
<summary>정답과 출력 해석</summary>

```python
selected = csv_df.loc[
    (csv_df["product"] == "A") | (csv_df["quantity"] == 1)
].copy()
print("선택된 행 수:", len(selected))

without_missing = csv_df.dropna(subset=["price"]).copy()
without_missing["amount"] = without_missing["price"] * without_missing["quantity"]
print("결측 행 제외:", without_missing["amount"].sum())
print("중앙값 대체:", clean["amount"].sum())
print("차이:", clean["amount"].sum() - without_missing["amount"].sum())
clean.to_csv(work / "orders_clean.csv", index=False, encoding="utf-8-sig")
```

선택 행 수는 3, 제외 합계는 118000, 대체 합계는 130000, 차이는 12000입니다. 상품 C의 수량은 1이고 결측 가격을 12000으로 가정하여 차이가 생깁니다. 데이터가 말해 준 실제 거래 가격은 아닙니다. 분석 결과에는 “가격 결측 1건을 중앙값으로 대체”라는 처리 기준을 함께 씁니다.

</details>

## 직접 실습 — 저장한 파일 다시 확인

`orders_clean.csv`를 다시 읽어 행 수와 열 목록을 출력합니다. `Unnamed: 0` 같은 불필요한 열이 없고 `amount` 열이 있는지 확인합니다.

<details>
<summary>정답 코드</summary>

```python
saved = pd.read_csv(work / "orders_clean.csv")
print(saved.shape)
print(saved.columns.tolist())
print(saved["amount"].sum())
```

예상 결과는 `(4, 4)`, `['product', 'price', 'quantity', 'amount']`, `130000.0`입니다. 앞 실습에서 행을 추가했다면 초기 4행 예제를 다시 실행한 뒤 비교합니다.

</details>

## 흔한 오류와 복구

| 오류·현상 | 확인과 해결 |
| --- | --- |
| `FileNotFoundError` | `Path.cwd()`와 읽으려는 경로의 `resolve()`를 확인합니다. `.csv`와 `.xlsx` 확장자를 구분합니다. |
| `ImportError`에 openpyxl 표시 | 수업 conda 환경에 openpyxl을 설치하고 Kernel을 재시작합니다. |
| `KeyError: price` | `csv_df.columns.tolist()`로 철자·대소문자·앞뒤 공백을 확인합니다. |
| 복합 조건에서 모호한 참값 오류 | pandas 조건에 `and/or` 대신 `&/|`를 사용하고 조건마다 괄호를 씁니다. |
| CSV에 `Unnamed: 0`이 생김 | 저장할 때 인덱스가 같이 저장되었습니다. 원래 표를 `index=False`로 다시 저장합니다. |
| 값이 `NaN` | 값을 모른다는 표시입니다. 실제 숫자 0과 구분합니다. |
| 결과가 예시와 다름 | 직접 실습에서 행을 추가했는지 확인하고 초기 데이터 생성 셀부터 다시 실행합니다. |

## 완료 체크

- [ ] 수업 폴더에서 uv로 Notebook을 열고 Kernel의 `sys.executable`이 수업 `.venv`인지 확인했습니다. [환경 안내](surl:14892)
- [ ] 실습을 저장한 뒤 Kernel을 재시작하고 선행 페이지의 준비 셀부터 위에서 아래로 다시 실행했습니다. 웹 수집은 저장한 자료와 성공·실패 기록을 구분했습니다.

- [ ] CSV·Excel에서 같은 크기의 DataFrame을 읽었습니다.
- [ ] Series와 한 열짜리 DataFrame을 구분했습니다.
- [ ] `loc` 라벨과 `iloc` 위치를 구분했습니다.
- [ ] 여러 조건을 괄호와 `&/|`로 결합했습니다.
- [ ] 결측치 처리 방식과 합계 차이를 함께 기록했습니다.
- [ ] 정리한 파일을 저장하고 다시 읽어 확인했습니다.

## 원자료와 이어서 보기

공식 안내문 3쪽의 CSV·Excel·행/열 선택·결측 정리 항목을 설명합니다. 이전 강사의 day1-2-분석.ipynb와 2장 PPT가 보조 자료입니다.

[수업 전체 안내](surl:14880) · [기존 단원: 데이터 수집 기초 및 pandas 활용](surl:14883) · [앞 단원: 파이썬 문법](surl:14882)

<!-- period-navigation:start -->
## 교시별 이동

[교시별 전체 안내](surl:14891) · [공식 단원별 안내](surl:14880) · [이전: 10월 6일 2~3교시 — 파이썬 문법과 DataFrame](surl:14893) · [다음: 10월 6일 6교시 — BeautifulSoup과 Selenium 웹 수집](surl:14895)
<!-- period-navigation:end -->